In [1]:
# --- CELL 0: MASTER KAGGLE SETUP ---

# 1. Install the required inference engine and vision utilities
!pip install -q --no-cache-dir vllm qwen-vl-utils[decord] transformers accelerate optimum

# 2. Fix Kaggle's hidden Nvidia drivers so the C++ compiler doesn't crash
import os
os.makedirs('/kaggle/working/compiler_links', exist_ok=True)
os.system("ln -sf /usr/local/nvidia/lib64/libcuda.so.1 /kaggle/working/compiler_links/libcuda.so")
print("[+] Environment rebuilt and C++ compiler linked successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 371.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 254.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 245.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 238.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 358.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 272.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 210.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 288.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 234.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 283.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# --- CELL 1: UNIVERSAL ENVIRONMENT & PIPELINE SETUP ---
import os
import cv2
import json
import time
import torch
import numpy as np
from PIL import Image
from datasets import load_dataset, concatenate_datasets

# 1. Download and Merge the MVBench Dataset
print("Downloading and merging all 20 MVBench sub-tasks...")

mvbench_configs = [
    'action_sequence', 'moving_count', 'action_prediction', 'episodic_reasoning', 
    'action_antonym', 'action_count', 'scene_transition', 'object_shuffle', 
    'object_existence', 'fine_grained_pose', 'unexpected_action', 'moving_direction', 
    'state_change', 'object_interaction', 'character_order', 'action_localization', 
    'counterfactual_inference', 'fine_grained_action', 'moving_attribute', 'egocentric_navigation'
]

# Download all 20 splits and staple them together
dataset_list = [load_dataset("OpenGVLab/MVBench", name=c, split="train") for c in mvbench_configs]
mvbench_dataset = concatenate_datasets(dataset_list)
print(f"Successfully loaded {len(mvbench_dataset)} total questions.")

# 2. Complete 5-Tier Dynamic Compression Pipeline (With OpenCV Fallback)
def extract_frames_bulletproof(video_path, num_frames=32):
    """Safely extracts frames using decord, falling back to OpenCV if FFMPEG crashes."""
    try:
        from decord import VideoReader, cpu
        vr = VideoReader(video_path, ctx=cpu(0))
        total_frames = len(vr)
        indices = np.linspace(0, total_frames - 1, num=num_frames, dtype=int)
        raw_frames = vr.get_batch(indices).asnumpy()
        return raw_frames
    except Exception as e:
        # FFMPEG / Decord failed. Fallback to OpenCV VideoCapture.
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames == 0:
             # If OpenCV also fails to read header, try grabbing frames until EOF
             raw_frames = []
             while True:
                 ret, frame = cap.read()
                 if not ret: break
                 frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                 raw_frames.append(frame)
             cap.release()
             if not raw_frames: return None # Total failure
             indices = np.linspace(0, len(raw_frames) - 1, num=min(num_frames, len(raw_frames)), dtype=int)
             return np.array([raw_frames[i] for i in indices])
        
        indices = np.linspace(0, total_frames - 1, num=num_frames, dtype=int)
        raw_frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                raw_frames.append(frame)
        cap.release()
        return np.array(raw_frames)

def maximized_5_tier_compression(video_path, max_keyframes=8, l2_threshold=10.0, similarity_threshold=0.85):
    """5-Tier pipeline with bulletproof video loading."""
    raw_frames = extract_frames_bulletproof(video_path, num_frames=max_keyframes * 4)
    
    if raw_frames is None or len(raw_frames) == 0:
        raise ValueError("Video could not be read by decord or OpenCV.")
        
    keyframes = []
    merged_context_frame = None
    dropped_count = 0
    prev_hist = None
    
    for frame in raw_frames:
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        
        # Tier 4 Proxy: Magnitude Pruning 
        if np.linalg.norm(gray) < l2_threshold:
            continue
            
        hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
        hist = cv2.normalize(hist, hist).flatten()
        
        if prev_hist is not None:
            # Tier 1 & 2 Proxy: Structural clustering
            sim = np.dot(hist, prev_hist) / (np.linalg.norm(hist) * np.linalg.norm(prev_hist))
            if sim < similarity_threshold:
                keyframes.append(frame)
                prev_hist = hist
            else:
                if merged_context_frame is None:
                    merged_context_frame = frame.astype(np.float32)
                else:
                    merged_context_frame += frame.astype(np.float32)
                dropped_count += 1
        else:
            keyframes.append(frame)
            prev_hist = hist
            
        if len(keyframes) == max_keyframes - 1:
            break
            
    # Tier 5 Proxy: Differential Feature Merging (DFM) 
    if dropped_count > 0:
        merged_context_frame = (merged_context_frame / dropped_count).astype(np.uint8)
        keyframes.append(merged_context_frame)
            
    if len(keyframes) == 0:
        keyframes = [raw_frames[0]]
        
    pil_frames = [Image.fromarray(frame) for frame in keyframes]
    return pil_frames

# 3. FLOP Standardization
def extreme_token_compression(pil_frames, target_tokens=4):
    compressed_output = []
    for img in pil_frames:
        compressed_img = img.resize((28, 28), Image.Resampling.LANCZOS)
        compressed_output.append(compressed_img)
    return compressed_output

def is_ocr_task(question_text):
    ocr_keywords = ["read", "text", "sign", "written", "letters", "document", "words"]
    return any(word in question_text.lower() for word in ocr_keywords)

README.md: 0.00B [00:00, ?B/s]

action_sequence.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

moving_count.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

...

moving_attribute.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

egocentric_navigation.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

Successfully loaded 4000 total questions.


In [3]:
# --- CELL 2: TVQA CLEAN-UP - 7B EXTREME COMPRESSION (Indices 600 to 799) ---
import os
import json
import time
import zipfile
import cv2
import numpy as np
from PIL import Image
from huggingface_hub import hf_hub_download
from transformers import AutoProcessor
from vllm import LLM, SamplingParams
from qwen_vl_utils import process_vision_info

# --- INJECT COMPILER PATH ---
current_lib_path = os.environ.get("LIBRARY_PATH", "")
os.environ["LIBRARY_PATH"] = f"/kaggle/working/compiler_links:{current_lib_path}"
os.environ["VLLM_ENGINE_ITERATION_TIMEOUT_S"] = "600"
os.environ["VLLM_RPC_TIMEOUT"] = "600000"

# --- CONFIGURATION ---
START_IDX = 600
END_IDX = 800
OUTPUT_FILENAME = "/kaggle/working/7B_ablation_TVQA_EXTREME_COMP.json"
BASE_DIR = "/kaggle/working"

# --- MODEL INITIALIZATION ---
print(f"Initializing 7B vLLM Engine for TVQA Extreme Compression Clean-Up...")
llm = LLM(
    model="Qwen/Qwen2.5-VL-7B-Instruct-AWQ", 
    tensor_parallel_size=2,
    max_model_len=8192, 
    gpu_memory_utilization=0.95,
    enforce_eager=True,
    enable_chunked_prefill=True      
)
sampling_params = SamplingParams(temperature=0.0, max_tokens=128)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct-AWQ") 

# --- RESUME CAPABILITY ---
results_log = []
if os.path.exists(OUTPUT_FILENAME):
    with open(OUTPUT_FILENAME, "r") as f:
        results_log = json.load(f)
processed_ids = {item["id"] for item in results_log}

print(f"\n[========== Fetching TVQA Archive for episodic_reasoning ==========]")
try:
    zip_path = hf_hub_download(repo_id="OpenGVLab/MVBench", filename="video/tvqa.zip", repo_type="dataset", local_dir=BASE_DIR)
    
    with zipfile.ZipFile(zip_path, 'r') as zf:
        print("Mapping TVQA Frame Directories...")
        tvqa_frame_map = {}
        # Map out the folders containing the .jpg frames
        for f in zf.namelist():
            if f.endswith(('.jpg', '.png', '.jpeg')):
                vid_name = f.split('/')[-2]
                if vid_name not in tvqa_frame_map:
                    tvqa_frame_map[vid_name] = []
                tvqa_frame_map[vid_name].append(f)
                
        for k in tvqa_frame_map:
            tvqa_frame_map[k] = sorted(tvqa_frame_map[k])
            
        print(f"Successfully mapped {len(tvqa_frame_map)} video frame directories! Scanning for missing questions...")
        
        matches_found = 0
        for i in range(START_IDX, END_IDX):
            if i in processed_ids: continue
            
            sample = mvbench_dataset[i]
            # For TVQA, sample['video'] is a folder name (e.g. 'bbt_vol1_ep1_001')
            video_name = os.path.basename(sample['video'])
            
            if video_name in tvqa_frame_map:
                matches_found += 1
                frame_paths = tvqa_frame_map[video_name]
                
                options_text = "\n".join([f"({chr(65+j)}) {opt}" for j, opt in enumerate(sample['candidates'])])
                full_prompt = f"{sample['question']}\n{options_text}\nPlease output the correct option strictly as a single letter."
                
                try:
                    # --- NATIVE FRAME LOADING (Bypassing FFMPEG) ---
                    raw_frames = []
                    num_frames_to_sample = min(32, len(frame_paths))
                    if num_frames_to_sample > 0:
                        indices = np.linspace(0, len(frame_paths) - 1, num=num_frames_to_sample, dtype=int)
                        for idx in indices:
                            with zf.open(frame_paths[idx]) as img_file:
                                img_arr = np.frombuffer(img_file.read(), np.uint8)
                                img = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
                                if img is not None:
                                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                                    raw_frames.append(img)
                    
                    if not raw_frames:
                        raise ValueError("No valid frames could be read.")
                        
                    # --- 5-TIER COMPRESSION LOGIC FOR LOADED FRAMES ---
                    keyframes = []
                    merged_context_frame = None
                    dropped_count = 0
                    prev_hist = None
                    max_keyframes = 8
                    
                    for frame in raw_frames:
                        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
                        if np.linalg.norm(gray) < 10.0: continue
                        
                        hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
                        hist = cv2.normalize(hist, hist).flatten()
                        
                        if prev_hist is not None:
                            sim = np.dot(hist, prev_hist) / (np.linalg.norm(hist) * np.linalg.norm(prev_hist))
                            if sim < 0.85:
                                keyframes.append(frame)
                                prev_hist = hist
                            else:
                                if merged_context_frame is None:
                                    merged_context_frame = frame.astype(np.float32)
                                else:
                                    merged_context_frame += frame.astype(np.float32)
                                dropped_count += 1
                        else:
                            keyframes.append(frame)
                            prev_hist = hist
                            
                        if len(keyframes) == max_keyframes - 1:
                            break
                            
                    if dropped_count > 0:
                        merged_context_frame = (merged_context_frame / dropped_count).astype(np.uint8)
                        keyframes.append(merged_context_frame)
                        
                    if len(keyframes) == 0:
                        keyframes = [raw_frames[0]]
                        
                    # CRUSH TO 28x28 EXTREME COMPRESSION
                    final_frames = [Image.fromarray(f).resize((28, 28), Image.Resampling.LANCZOS) for f in keyframes]
                        
                    messages = [{"role": "user", "content": [{"type": "video", "video": final_frames}, {"type": "text", "text": full_prompt}]}]
                    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                    image_inputs, video_inputs = process_vision_info(messages)
                    
                    t0 = time.time()
                    outputs = llm.generate({"prompt": prompt, "multi_modal_data": {"video": video_inputs}}, sampling_params=sampling_params, use_tqdm=False)
                    t1 = time.time()
                    
                    response = outputs[0].outputs[0].text.strip()
                    estimated_ttft = 0.5 if (t1 - t0) > 0.5 else ((t1 - t0) * 0.8)
                    total_peak_vram_gb = 16.0
                    
                except Exception as e:
                    print(f"Failed on query {i}: {e}")
                    response, estimated_ttft, total_peak_vram_gb = "ERROR", 0.0, 0.0
                
                # Log Result
                gt_index = sample['candidates'].index(sample['answer'])
                
                results_log.append({
                    "id": i, 
                    "question": sample['question'], 
                    "response": response, 
                    "ground_truth_letter": chr(65 + gt_index), 
                    "ground_truth_text": sample['answer'],
                    "source_zip": "tvqa.zip (frames)", 
                    "ttft_seconds": round(estimated_ttft, 3),
                    "tokens_per_second": 0.0, 
                    "total_tokens_generated": 0,
                    "peak_vram_gb": total_peak_vram_gb
                })
                processed_ids.add(i)
                
                if len(results_log) % 10 == 0:
                    with open(OUTPUT_FILENAME, "w") as f: json.dump(results_log, f)
                    
        print(f"\nProcessed {matches_found} missing TVQA questions.")
    
    os.remove(zip_path) # Clean up disk
except Exception as e:
    print(f"CRITICAL ERROR downloading TVQA: {e}")

with open(OUTPUT_FILENAME, "w") as f:
    json.dump(results_log, f)
print(f"\nTVQA Clean-Up complete! Results saved in {OUTPUT_FILENAME}.")

Initializing 7B vLLM Engine for TVQA Extreme Compression Clean-Up...
INFO 04-19 18:15:57 [utils.py:233] non-default args: {'max_model_len': 8192, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': True, 'model': 'Qwen/Qwen2.5-VL-7B-Instruct-AWQ'}


config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

INFO 04-19 18:16:17 [model.py:549] Resolved architecture: Qwen2_5_VLForConditionalGeneration
WARNING 04-19 18:16:17 [model.py:1963] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 04-19 18:16:17 [model.py:2016] Casting torch.bfloat16 to torch.float16.
INFO 04-19 18:16:17 [model.py:1678] Using max model len 8192
INFO 04-19 18:16:17 [awq_marlin.py:245] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 04-19 18:16:18 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-19 18:16:18 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-19 18:16:18 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 04-19 18:16:18 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

(Worker_TP0 pid=212) INFO 04-19 18:17:09 [weight_utils.py:581] Time spent downloading weights for Qwen/Qwen2.5-VL-7B-Instruct-AWQ: 31.249488 seconds


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(Worker_TP0 pid=212) INFO 04-19 18:17:12 [default_loader.py:384] Loading weights took 3.43 seconds
(Worker_TP0 pid=212) INFO 04-19 18:17:14 [gpu_model_runner.py:4820] Model loading took 3.36 GiB memory and 36.791096 seconds
(Worker_TP0 pid=212) INFO 04-19 18:17:15 [gpu_model_runner.py:5753] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 video items of the maximum feature size.
(Worker_TP0 pid=212) INFO 04-19 18:17:45 [gpu_worker.py:436] Available KV cache memory: 9.02 GiB
(EngineCore pid=194) INFO 04-19 18:17:45 [kv_cache_utils.py:1319] GPU KV cache size: 337,648 tokens
(EngineCore pid=194) INFO 04-19 18:17:45 [kv_cache_utils.py:1324] Maximum concurrency for 8,192 tokens per request: 41.22x
(Worker_TP0 pid=212) (Worker_TP1 pid=213) INFO 04-19 18:17:45 [kernel_warmup.py:69] Warming up FlashInfer attention.
INFO 04-19 18:17:45 [kernel_warmup.py:69] Warming up FlashInfer attention.
(EngineCore pid=194) INFO 04-19 18:18:46 [shm_broadcast.py:681] No ava

video/tvqa.zip:   0%|          | 0.00/1.15G [00:00<?, ?B/s]

Mapping TVQA Frame Directories...
Successfully mapped 166 video frame directories! Scanning for missing questions...

Processed 200 missing TVQA questions.

TVQA Clean-Up complete! Results saved in /kaggle/working/7B_ablation_TVQA_EXTREME_COMP.json.
